# IPL Feature Engineering — Full Pipeline
Run cells top to bottom. Outputs a clean `features.parquet` ready for model training.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from feature_engineering import (
    build_features,
    match_level_split,
    encode_teams_on_train,
    feature_summary,
    FEATURE_COLS,
    TARGET_COL,
    ID_COL,
)

sns.set_theme(style='darkgrid', palette='muted')

In [ ]:
# ── Run the pipeline ──────────────────────────────────────
df = build_features('data/matches.csv', 'data/deliveries.csv')
df.head(3)

In [ ]:
# ── Feature summary stats ────────────────────────────────
feature_summary(df)

In [ ]:
# ── Win probability by over (sanity check) ───────────────
win_by_over = df.groupby('over')[TARGET_COL].mean().reset_index()
plt.figure(figsize=(10, 4))
plt.plot(win_by_over['over'], win_by_over[TARGET_COL] * 100, marker='o', lw=2)
plt.axhline(50, color='gray', linestyle='--', lw=1)
plt.xlabel('Over'); plt.ylabel('Average win % for batting team')
plt.title('Win rate by over — should be ~50% at start, varies by over 20')
plt.tight_layout(); plt.show()

In [ ]:
# ── RRR distribution (should peak around 6–9) ───────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
df['rrr'].clip(0, 25).hist(bins=40, ax=axes[0], color='steelblue')
axes[0].set_title('Required run rate distribution')

df['momentum_6'].hist(bins=40, ax=axes[1], color='seagreen')
axes[1].set_title('Momentum (last 6 balls)')

df['partnership_runs'].clip(0, 120).hist(bins=40, ax=axes[2], color='coral')
axes[2].set_title('Partnership runs')

plt.tight_layout(); plt.show()

In [ ]:
# ── Correlation heatmap (numeric features only) ──────────
num_cols = [c for c in FEATURE_COLS if c not in ('bat_enc', 'bowl_enc')] + [TARGET_COL]
corr = df[num_cols].corr()[[TARGET_COL]].drop(TARGET_COL).sort_values(TARGET_COL)

plt.figure(figsize=(5, 8))
sns.barh(corr.index, corr[TARGET_COL], color=[
    'seagreen' if v > 0 else 'coral' for v in corr[TARGET_COL]
])
plt.axvline(0, color='gray', lw=0.8)
plt.xlabel('Pearson correlation with win')
plt.title('Feature correlation with target')
plt.tight_layout(); plt.show()

In [ ]:
# ── Match-level split ────────────────────────────────────
train_df, test_df = match_level_split(df, test_size=0.2, random_state=42)

In [ ]:
# ── Encode teams on train only (no leakage) ──────────────
train_df, test_df, le_bat, le_bowl = encode_teams_on_train(train_df, test_df)

print('Teams in encoder:', list(le_bat.classes_))

In [ ]:
# ── Final feature matrices ───────────────────────────────
X_train = train_df[FEATURE_COLS].fillna(0)
y_train = train_df[TARGET_COL]
X_test  = test_df[FEATURE_COLS].fillna(0)
y_test  = test_df[TARGET_COL]

print(f'X_train: {X_train.shape}  |  X_test: {X_test.shape}')
print(f'y_train win rate: {y_train.mean():.2%}  |  y_test: {y_test.mean():.2%}')

In [ ]:
# ── Save for model training ──────────────────────────────
import pickle, os
os.makedirs('model', exist_ok=True)

X_train.to_parquet('data/X_train.parquet', index=False)
y_train.to_frame().to_parquet('data/y_train.parquet', index=False)
X_test.to_parquet('data/X_test.parquet', index=False)
y_test.to_frame().to_parquet('data/y_test.parquet', index=False)

pickle.dump(le_bat,  open('model/le_bat.pkl',  'wb'))
pickle.dump(le_bowl, open('model/le_bowl.pkl', 'wb'))

print('Saved X_train, y_train, X_test, y_test + encoders.')